# Samsung PRISM 26VI05 - full pipeline on Colab
1. Runtime -> Change runtime type -> **GPU** (L4/A100 best, T4 works). A CPU runtime can generate images but cannot train.
2. Runtime -> Run all. The code is cloned from GitHub into `MyDrive/prism_ladder`; images, checkpoints and reports
   are saved there too, so after a disconnect just Run all again and every stage resumes where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')
if not os.path.exists('prism_ladder'):
    !git clone --depth 1 https://github.com/sansak-123/Samsung-prism.git prism_ladder
os.chdir('/content/drive/MyDrive/prism_ladder')
!git checkout -- configs/config.yaml && git pull --ff-only
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q transformers==4.51.3 trl==0.17.0 peft==0.15.2 'accelerate>=1.6.0' 'bitsandbytes>=0.45.0' 'datasets>=3.0' qwen-vl-utils==0.0.11 scikit-image openpyxl pyyaml

In [ ]:
# Low-VRAM GPU (T4, 15 GB): smaller images so training fits
import torch, yaml
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
cfg = yaml.safe_load(open('configs/config.yaml'))
if mem < 20:
    cfg['model']['max_pixels'] = 501760; cfg['inference']['batch_size'] = 2
    yaml.safe_dump(cfg, open('configs/config.yaml', 'w'), sort_keys=False)
print(f'{mem:.0f} GB ->', cfg['model']['max_pixels'], cfg['inference']['batch_size'])

In [ ]:
import json
if not os.path.exists('data/metadata/ladder.jsonl'):
    !python scripts/01_generate_dataset.py
!python scripts/02_build_splits.py
if not os.path.exists('pilot/images'):
    !python scripts/make_pilot_showcase.py
!python scripts/00_selftest.py

In [ ]:
!python scripts/gpu_smoke_test.py

In [ ]:
!python scripts/03_prompt_study_and_baseline.py

In [ ]:
!python scripts/04_train_qlora.py

In [ ]:
!python scripts/05_infer.py --adapter outputs/best_adapter --name finetuned_qlora --split test
!python scripts/06_evaluate.py --models baseline_zero_shot finetuned_qlora --split test

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/evaluation_report.md').read()))
display(Image('reports/figures/mae_per_metric.png'))